In [1]:
import streamlit as st
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import tensorflow as tf
import os

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'autoint').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')

ROOT_DIR = find_project_root()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import joblib
from autoint.autoint import AutoIntModel, predict_model
from autoint.paths import data_dir, model_dir
from autoint.tf_device import configure_tensorflow

configure_tensorflow()

In [2]:
data_path = data_dir()
movielens_dir_nm = 'ml-1m'
model_path = model_dir()
field_dims = np.load(data_path / 'field_dims.npy')
dropout= 0.4
embed_dim= 16

In [10]:
ratings_df = pd.read_csv(f'{data_path}/{movielens_dir_nm}/ratings_prepro.csv')
movies_df = pd.read_csv(f'{data_path}/{movielens_dir_nm}/movies_prepro.csv')
user_df = pd.read_csv(f'{data_path}/{movielens_dir_nm}/users_prepro.csv')

model = AutoIntModel(field_dims, embed_dim, att_layer_num=3, att_head_num=2, att_res=True,
                             l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=dropout, init_std=0.0001)
model(tf.constant([[0] * len(field_dims)], dtype=tf.int64))

model.load_weights(f'{model_path}/autoInt_model_weights.weights.h5')
label_encoders = joblib.load(f'{data_path}/label_encoders.pkl')

c:\Users\Admin\rec_node\venv\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
